In [1]:
%load_ext autoreload
%autoreload 2

import os, sys
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path

# coastsat modules
# sys.path.insert(0, os.pardir)
# from coastsat import SDS_download, SDS_preprocess, SDS_shoreline, SDS_tools, SDS_classify, SDS_transects
from osgeo import gdal
gdal.UseExceptions()

from indices import *
from thresholding import *
# from spectral_plotting import plot_extractions
# import wv_utils as wv
# import wv_eval
# import wv_eval_plotting as wvep
# import modified_coastsat
# import metrics as m
# import local_sl_extraction as lse
# import spectral_unmixing as su

# paths
base_dir = "C:\\Users\\avanever\\Documents\\Imagery Datasets\\WV3 SWIR\\W075_N10\\20200104T15431"
swir_fn = os.path.join(base_dir, "20JAN04154316-A3DS-015384188020_01_P001.TIF")
ms_fn = os.path.join(base_dir, "20JAN04154317-M3DS-015384188010_01_P001.TIF")
pan_fn = os.path.join(base_dir, "20JAN04154317-P3DS-015384188010_01_P001.TIF")
ref_sl_path = Path(r"C:\Users\avanever\Documents\CoastSatProject\Experiments\data\colombia_hand_digitized\colombia_hand_shoreline.shp")

ImportError: DLL load failed: The specified module could not be found.

In [2]:
# both transects and shoreline are in EPSG32618 - no conversions necessary

# load low res sim image
sim_fn = wv.get_sim_tif_name("naive_res_10", base_dir)
sim = gdal.Open(sim_fn, gdal.GA_ReadOnly)
georef = sim.GetGeoTransform()
image_epsg = 32618

sim_bands = wv.load_sim_tif(sim_fn)
im_rgb = sim_bands[:,:,[2, 1, 0]]

# load transects
transect_fn = Path(r"C:\Users\avanever\Documents\CoastSatProject\Experiments\data\transects\colombia_transects.geojson")
transects = np.array(list(SDS_tools.transects_from_geojson(transect_fn).values()))
print(f"{transects.shape = }")

# load reference shoreline
ref_sl_path = Path(r"C:\Users\avanever\Documents\CoastSatProject\Experiments\data\colombia_hand_digitized\colombia_hand_shoreline.shp")
ref_sl_list = wv.load_ref_sl(ref_sl_path)
ref_sl_pxl_list = wv.ref_sl_to_pxl(ref_sl_list, georef)
ref_sl_points = wv.shoreline_to_points(ref_sl_list, delta=0.5)
sl_buffer = wv.create_shoreline_buffer(sim_bands.shape[:-1], ref_sl_pxl_list, 10)
print(f"{ref_sl_points.shape = }")

# initialize empty cloud mask
im_nodata = np.full(sim_bands.shape[:-1], False)
cloud_mask = np.full(sim_bands.shape[:-1], False)

# convert transects and reference shoreline to pixel space
transects_0_pxl = SDS_tools.convert_world2pix(transects[:,0,:], georef)
transects_1_pxl = SDS_tools.convert_world2pix(transects[:,1,:], georef)
transects_pix = np.swapaxes(np.stack([transects_0_pxl, transects_1_pxl]), 0, 1)
ref_sl_points_pxl = SDS_tools.convert_world2pix(ref_sl_points, georef)

# data/settings dictionaries
sds_settings = {
    'output_epsg': image_epsg,
    'max_dist_ref': 400,
    'min_length_sl': 150, # 500
    'dist_clouds': 30,
}

collider_settings = dict(
    past_dist = 10,
    along_dist = 25,
    min_chainage = -10
)

sds_data = dict(
    cloud_mask=cloud_mask,
    sl_buffer=sl_buffer,
    im_nodata=im_nodata,
    georef=georef,
    image_epsg=image_epsg,
    sds_settings=sds_settings
)

86 transects have been loaded 
coordinates are in epsg:4326
transects.shape = (86, 2, 2)
ref_sl_points.shape = (22900, 2)


In [3]:
index_functions = [
    mndwi, ndwi, awei_ns, awei_sh, scowi,
    wi2015, andwi, wri, ewi, nwi, wi2019,
    tct_wetness, ddwi, ensemble1, ensemble2,
]
threshold_functions = [
    otsu,
    local_min_otsu
]
ensemble_idx_funcs = index_functions[:-2] # all but the ensemble indices